# ⚙️ 02 — Prétraitement & Clustering — Social Segmentation

**Objectif** : Normaliser les données nettoyées, choisir le k optimal et appliquer K-Means.

**Entrée** : `data/processed/Segmentation Data _clean.csv` (produit par le notebook 01)  
**Sortie** : `data/processed/segmentation_clustered.csv` + graphiques dans `results/`

---

## 0️⃣ Chemins & Imports

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples
from scipy import stats
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

ROOT           = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PROCESSED = ROOT / 'data' / 'processed'
RESULTS        = ROOT / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

# Chargement du fichier nettoyé (notebook 01)
csvs = sorted(DATA_PROCESSED.glob('*_clean.csv'))
if not csvs:
    raise FileNotFoundError('Lancer d\'abord 01_eda_social_segmentation.ipynb')
CLEAN_PATH = csvs[0]
print('Fichier chargé :', CLEAN_PATH.name)

df = pd.read_csv(CLEAN_PATH)
display(df.head())
print('Dimensions :', df.shape)

## 1️⃣ Normalisation (StandardScaler)

In [ ]:
FEATURES = ['Age', 'Gender_enc', 'Income', 'Score']

X       = df[FEATURES].values
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X)

df_scaled = pd.DataFrame(X_scaled, columns=FEATURES)
print('Avant normalisation :')
display(df[FEATURES].describe().round(2))
print('\nAprès normalisation (StandardScaler) :')
display(df_scaled.describe().round(3))

## 2️⃣ Choix du k optimal — Elbow + Silhouette

In [ ]:
K_MIN, K_MAX = 2, 10
k_range      = range(K_MIN, K_MAX + 1)
inertias     = []
silhouettes  = []

for k in tqdm(k_range, desc='Test K-Means'):
    km     = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

# ── Tableau récapitulatif ─────────────────────────────────────────────────────
summary = pd.DataFrame({'k': list(k_range),
                         'Inertie': inertias,
                         'Silhouette': silhouettes})
display(summary.set_index('k').round(4))

# ── Graphique statique ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(k_range), inertias,  'o-', color='#2ecc71', lw=2, markersize=7)
axes[0].set_title('Méthode Elbow (Inertie)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertie'); axes[0].grid(alpha=0.3)

axes[1].plot(list(k_range), silhouettes, 's-', color='#e74c3c', lw=2, markersize=7)
axes[1].set_title('Score Silhouette', fontsize=13, fontweight='bold')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette'); axes[1].grid(alpha=0.3)

best_k = list(k_range)[int(np.argmax(silhouettes))]
axes[1].axvline(best_k, color='navy', linestyle='--', lw=2, label=f'Meilleur k={best_k}')
axes[1].legend()
plt.tight_layout()
plt.savefig(RESULTS / 'clustering_elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✔ Meilleur k = {best_k}')

## 3️⃣ Application K-Means avec k optimal

In [ ]:
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['Cluster'] = km_final.fit_predict(X_scaled)

sil_final = silhouette_score(X_scaled, df['Cluster'])
print(f'Silhouette Score final : {sil_final:.4f}')
print('\nTaille des clusters :')
print(df['Cluster'].value_counts().sort_index().to_frame('Taille'))

## 4️⃣ Analyse Silhouette par cluster

In [ ]:
sample_sil = silhouette_samples(X_scaled, df['Cluster'])
palette    = sns.color_palette('tab10', best_k)

fig, ax = plt.subplots(figsize=(9, 6))
y_lower = 10
for c in range(best_k):
    vals   = np.sort(sample_sil[df['Cluster'] == c])
    size_c = len(vals)
    y_upper = y_lower + size_c
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, vals,
                     facecolor=palette[c], alpha=0.75, label=f'C{c}')
    ax.text(-0.05, y_lower + size_c / 2, f'C{c}', fontsize=10)
    y_lower = y_upper + 10

ax.axvline(sil_final, color='red', linestyle='--', lw=2,
           label=f'Score moyen = {sil_final:.3f}')
ax.set_title('Analyse Silhouette par cluster', fontsize=13, fontweight='bold')
ax.set_xlabel('Coefficient Silhouette')
ax.set_ylabel('Cluster')
ax.legend(loc='lower right'); ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(RESULTS / 'clustering_silhouette_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 5️⃣ Projection PCA 2D (statique + interactif)

In [ ]:
pca   = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
var   = pca.explained_variance_ratio_

df['PC1'] = X_pca[:, 0]
df['PC2'] = X_pca[:, 1]

# Statique
fig, ax = plt.subplots(figsize=(9, 7))
for c in range(best_k):
    m = df['Cluster'] == c
    ax.scatter(df.loc[m, 'PC1'], df.loc[m, 'PC2'],
               label=f'Cluster {c}', color=palette[c],
               alpha=0.7, edgecolors='white', s=60)
ax.set_title('Clustering K-Means — PCA 2D', fontsize=14, fontweight='bold')
ax.set_xlabel(f'PC1 ({var[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({var[1]*100:.1f}% variance)')
ax.legend(title='Cluster'); ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(RESULTS / 'clustering_pca2d.png', dpi=150, bbox_inches='tight')
plt.show()

# Interactif Plotly
fig_px = px.scatter(
    df, x='PC1', y='PC2',
    color=df['Cluster'].astype(str),
    hover_data=['Age', 'Gender', 'Income', 'Score'],
    title='Clustering — PCA 2D interactif',
    labels={'color': 'Cluster'},
    template='plotly_white'
)
fig_px.write_html(str(RESULTS / 'clustering_pca2d_interactive.html'))
fig_px.show()
print('✔ PCA 2D interactif sauvegardé')

## 6️⃣ Profiling des clusters

In [ ]:
profile = df.groupby('Cluster').agg(
    Taille       = ('Cluster', 'count'),
    Age_moyen    = ('Age',    'mean'),
    Age_std      = ('Age',    'std'),
    Revenu_moyen = ('Income', 'mean'),
    Revenu_std   = ('Income', 'std'),
    Score_moyen  = ('Score',  'mean'),
    Score_std    = ('Score',  'std'),
    Pct_hommes   = ('Gender_enc', lambda x: round(x.mean() * 100, 1)),
).round(2)

display(profile)

# Heatmap des profils
heat_data = profile[['Age_moyen', 'Revenu_moyen', 'Score_moyen']]
heat_norm = (heat_data - heat_data.min()) / (heat_data.max() - heat_data.min())

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(heat_norm, annot=heat_data.round(0).astype(int), fmt='d',
            cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title('Profil moyen des clusters (normalisé)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS / 'clustering_heatmap_profils.png', dpi=150, bbox_inches='tight')
plt.show()

## 7️⃣ Scatter 3D interactif — Age / Income / Score

In [ ]:
fig_3d = px.scatter_3d(
    df, x='Age', y='Income', z='Score',
    color=df['Cluster'].astype(str),
    symbol='Gender', opacity=0.75,
    title='Segmentation 3D : Âge / Revenu / Score',
    labels={'color': 'Cluster'},
    template='plotly_white'
)
fig_3d.write_html(str(RESULTS / 'clustering_scatter3d.html'))
fig_3d.show()
print('✔ Scatter 3D interactif sauvegardé')

## 8️⃣ Sauvegarde du fichier clustérisé

In [ ]:
out_path = DATA_PROCESSED / 'segmentation_clustered.csv'
df.to_csv(out_path, index=False)

print(f'✔ Fichier sauvegardé : {out_path.resolve()}')
print(f'  Lignes : {len(df)} | Colonnes : {list(df.columns)}')
display(df.head())

## ✅ Synthèse Clustering

| Paramètre | Valeur |
|-----------|--------|
| **Algorithme** | K-Means |
| **k optimal** | À compléter après exécution |
| **Silhouette Score** | À compléter après exécution |
| **Features** | Age, Gender_enc, Income, Score |
| **Normalisation** | StandardScaler |

**Prochaine étape** → `03_rapport_final.ipynb`  
Conclusions, recommandations métier, export Excel